# OpenAI Agents SDK: 첫 번째 에이전트 프레임워크 프로젝트

이번 노트북에서는 **OpenAI Agents SDK**를 활용하여 멀티 에이전트 시스템을 구축합니다. 이전 노트북에서 직접 구현했던 Tool Use와 에이전트 루프를, 프레임워크가 어떻게 간소화하는지 비교하며 학습합니다.

## 개요

| 주제 | 내용 |
|------|------|
| Agents SDK 소개 | OpenAI의 에이전트 프레임워크 핵심 개념 |
| 에이전트 워크플로우 | 여러 에이전트를 생성하고 병렬 실행 |
| @function_tool | 데코레이터 기반 도구 정의 (JSON 스키마 자동 생성) |
| 에이전트를 도구로 | as_tool()로 에이전트 간 협업 |
| Handoff | 에이전트 간 제어 전달 |
| Trace | OpenAI 트레이스로 실행 흐름 모니터링 |

## 학습 목표

1. OpenAI Agents SDK의 핵심 개념(Agent, Runner, Tool, Handoff) 이해하기
2. 여러 에이전트를 병렬로 실행하고 결과를 비교/선택하기
3. `@function_tool` 데코레이터로 간편하게 도구 정의하기
4. 에이전트 간 협업 패턴(도구 vs 핸드오프) 비교하기
5. Trace를 활용한 에이전트 실행 흐름 모니터링

---

## 이전 노트북과의 비교

```
┌─────────────────────────────────────────────────────────────────────┐
│               직접 구현 vs Agents SDK 비교                         │
├──────────────────────────────┬──────────────────────────────────────┤
│     이전 (직접 구현)          │     이번 (Agents SDK)               │
├──────────────────────────────┼──────────────────────────────────────┤
│  JSON 스키마 직접 작성        │  @function_tool 데코레이터 사용      │
│  handle_tool_calls() 구현    │  SDK가 자동 처리                     │
│  while 루프 직접 관리         │  Runner.run()이 자동 실행            │
│  메시지 히스토리 수동 관리    │  SDK가 컨텍스트 자동 관리            │
│  에이전트 간 통신 직접 구현   │  Handoff로 간편하게 위임             │
└──────────────────────────────┴──────────────────────────────────────┘
```

---

## 1. OpenAI Agents SDK 핵심 개념

OpenAI Agents SDK는 몇 가지 핵심 프리미티브로 구성되어 있습니다:

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Agents SDK 핵심 구성 요소                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────┐   instructions + tools + handoffs를 가진 LLM          │
│  │  Agent   │   에이전트는 자체 시스템 프롬프트와 도구를 보유        │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   Python 함수에 @function_tool을 붙여서 생성          │
│  │  Tool    │   JSON 스키마가 자동으로 생성됨                       │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트 간 제어를 전달하는 메커니즘                 │
│  │ Handoff  │   도구: 제어가 돌아옴 / 핸드오프: 제어가 넘어감      │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트를 실행하고 결과를 반환                      │
│  │ Runner   │   run(), run_streamed() 등 다양한 실행 방식          │
│  └─────────┘                                                       │
│                                                                     │
│  ┌─────────┐   에이전트 실행 흐름을 추적하고 시각화                 │
│  │  Trace   │   OpenAI 대시보드에서 확인 가능                      │
│  └─────────┘                                                       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Agent 간 협업: 도구(Tool) vs 핸드오프(Handoff)

| 방식 | 제어 흐름 | 사용 시점 |
|------|----------|----------|
| **도구(as_tool)** | 호출 후 제어가 **돌아옴** | 다른 에이전트의 결과를 받아서 추가 처리할 때 |
| **핸드오프(Handoff)** | 제어가 다른 에이전트로 **넘어감** | 작업을 완전히 위임할 때 |

```
도구 (Tool):     A ──호출──▶ B ──결과──▶ A (제어가 A로 복귀)
핸드오프 (Handoff): A ──위임──▶ B (제어가 B로 이전)
```

---

## 2. 환경 설정

In [ ]:
# OpenAI Agents SDK 설치 (처음 한 번만 실행)
# 터미널에서: uv add openai-agents
# 또는 노트북에서:
# import sys
# !{sys.executable} -m pip install openai-agents

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import asyncio
import os

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

---

## 3. 에이전트 워크플로우

동일한 작업을 **서로 다른 성격**의 에이전트에게 맡겨보겠습니다.

시나리오: AI 기반 고객 서비스 자동화 SaaS 회사 **"AiDesk"**의 영업 이메일을 작성합니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                    3명의 영업 에이전트                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐         │
│  │  전문적 스타일 │  │  유머 스타일  │  │  간결 스타일  │         │
│  │  (격식체)     │  │  (친근체)     │  │  (핵심만)    │         │
│  └──────────────┘  └──────────────┘  └──────────────┘         │
│         │                 │                 │                   │
│         └─────────────────┼─────────────────┘                   │
│                           ▼                                     │
│                    동일한 입력 메시지                            │
│                    "콜드 영업 이메일 작성"                       │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# 3가지 스타일의 영업 에이전트 정의

instructions1 = """당신은 AiDesk의 영업 담당자입니다.
AiDesk는 AI 기반 고객 서비스 자동화 SaaS 플랫폼으로, 
고객 문의 자동 응답, 티켓 분류, 감정 분석 기능을 제공합니다.
당신은 전문적이고 격식 있는 콜드 영업 이메일을 작성합니다."""

instructions2 = """당신은 AiDesk의 영업 담당자입니다.
AiDesk는 AI 기반 고객 서비스 자동화 SaaS 플랫폼으로,
고객 문의 자동 응답, 티켓 분류, 감정 분석 기능을 제공합니다.
당신은 유머러스하고 친근한 콜드 영업 이메일을 작성합니다.
받는 사람이 답장하고 싶어지는 재치 있는 문체를 사용합니다."""

instructions3 = """당신은 AiDesk의 영업 담당자입니다.
AiDesk는 AI 기반 고객 서비스 자동화 SaaS 플랫폼으로,
고객 문의 자동 응답, 티켓 분류, 감정 분석 기능을 제공합니다.
당신은 바쁜 영업 담당자로서, 간결하고 핵심만 담은 콜드 영업 이메일을 작성합니다."""

### Agent 클래스 이해하기

`Agent`는 OpenAI Agents SDK의 가장 기본적인 구성 요소입니다. **LLM + 지시사항 + 도구**를 하나로 묶은 객체입니다.

```python
Agent(
    name="에이전트 이름",           # 식별용 이름 (트레이스에서 표시됨)
    instructions="시스템 프롬프트",  # 에이전트의 역할과 행동 규칙 (str 또는 callable)
    model="gpt-4o-mini",           # 사용할 모델
    tools=[...],                   # 사용할 도구 목록 (@function_tool 또는 as_tool())
    handoffs=[...],                # 제어를 위임할 다른 에이전트 목록
    model_settings=ModelSettings(  # 모델 상세 설정 (선택)
        temperature=0.7,
        top_p=1.0,
    ),
    output_type=MyModel,           # 구조화된 출력 타입 - Pydantic 모델 (선택)
)
```

| 파라미터 | 필수 | 설명 |
|----------|------|------|
| `name` | O | 에이전트 식별 이름. 트레이스와 핸드오프에서 사용됨 |
| `instructions` | O | 시스템 프롬프트. 문자열 또는 `callable`(동적 생성) 가능 |
| `model` | - | 사용할 LLM 모델. 기본값은 SDK 설정에 따름 |
| `tools` | - | `@function_tool` 함수 또는 `agent.as_tool()` 목록 |
| `handoffs` | - | 제어를 넘길 수 있는 다른 Agent 객체 목록 |
| `model_settings` | - | temperature, top_p 등 모델 세부 설정 |
| `output_type` | - | Pydantic 모델로 구조화된 출력 강제 |

> **이전 노트북과 비교**: 직접 구현할 때는 시스템 프롬프트, 모델명, 도구 스키마를 각각 별도 변수로 관리했지만, `Agent` 클래스가 이를 **하나의 객체로 캡슐화**합니다.

In [ ]:
# Agent 객체 생성 — 이전 노트북의 시스템 프롬프트 + 모델 설정과 동일한 역할

sales_agent1 = Agent(
    name="전문적 영업 에이전트",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="유머 영업 에이전트",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="간결 영업 에이전트",
    instructions=instructions3,
    model="gpt-4o-mini"
)

print(f"에이전트 생성 완료: {sales_agent1.name}, {sales_agent2.name}, {sales_agent3.name}")

### 3.1 단일 에이전트 실행 (스트리밍)

In [ ]:
# 스트리밍 방식으로 에이전트 실행
# Runner.run_streamed()는 토큰을 실시간으로 받아볼 수 있습니다

result = Runner.run_streamed(sales_agent1, input="콜드 영업 이메일을 작성해주세요.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

### 3.2 병렬 실행: 3개 에이전트 동시 실행

이전 노트북에서 학습한 **병렬화(Parallelization)** 패턴을 Agents SDK로 구현합니다.

`asyncio.gather()`를 사용하여 3개의 에이전트를 **동시에** 실행합니다.

In [ ]:
message = "콜드 영업 이메일을 작성해주세요."

# trace()로 실행 흐름을 기록합니다 — OpenAI 대시보드에서 확인 가능
with trace("병렬 콜드 이메일 생성"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

labels = ["전문적 스타일", "유머 스타일", "간결 스타일"]
for label, output in zip(labels, outputs):
    print(f"{'='*60}")
    print(f"[{label}]")
    print(f"{'='*60}")
    print(output)
    print()

### 3.3 최적 이메일 선택 (Voting 패턴)

3개의 이메일 중 **가장 효과적인 이메일**을 선택하는 에이전트를 추가합니다.

이것은 이전 노트북에서 배운 **Parallelization → Voting** 패턴의 구현입니다.

```
┌──────────────┐  ┌──────────────┐  ┌──────────────┐
│  에이전트 1   │  │  에이전트 2   │  │  에이전트 3   │
│  이메일 생성  │  │  이메일 생성  │  │  이메일 생성  │
└──────┬───────┘  └──────┬───────┘  └──────┬───────┘
       │                 │                 │
       └─────────────────┼─────────────────┘
                         ▼
              ┌──────────────────────┐
              │   선택 에이전트      │
              │  (최적 이메일 선별)   │
              └──────────┬───────────┘
                         ▼
                   최종 선택 이메일
```

In [ ]:
# 이메일 선택 에이전트 — 고객 관점에서 가장 좋은 이메일을 고릅니다

sales_picker = Agent(
    name="이메일 선택 에이전트",
    instructions="""주어진 콜드 영업 이메일 후보들 중에서 가장 좋은 것을 선택하세요.
당신이 고객이라고 상상하고, 가장 답장하고 싶은 이메일을 고르세요.
설명 없이 선택한 이메일 본문만 출력하세요.""",
    model="gpt-4o-mini"
)

In [ ]:
message = "콜드 영업 이메일을 작성해주세요."

with trace("최적 영업 이메일 선택"):
    # Step 1: 3개의 이메일을 병렬 생성
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    # Step 2: 선택 에이전트에게 전달
    emails = "콜드 영업 이메일 후보들:\n\n" + "\n\n---이메일---\n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)

    print("최적 영업 이메일:")
    print("=" * 60)
    print(best.final_output)

### Trace 확인하기

`trace()`로 감싼 실행은 OpenAI 대시보드에서 시각적으로 확인할 수 있습니다:

https://platform.openai.com/traces

트레이스를 확인하면 각 에이전트가 어떤 순서로, 얼마나 걸려서 실행되었는지 확인할 수 있습니다.

---

## 4. @function_tool로 도구 정의하기

이전 노트북에서는 도구를 정의할 때 **JSON 스키마를 직접 작성**해야 했습니다.

Agents SDK에서는 `@function_tool` 데코레이터만 붙이면 **자동으로 JSON 스키마가 생성**됩니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│                  도구 정의 방식 비교                                │
├────────────────────────────────┬────────────────────────────────────┤
│     이전 (직접 JSON 스키마)     │     이번 (@function_tool)         │
├────────────────────────────────┼────────────────────────────────────┤
│  1. Python 함수 작성           │  1. Python 함수에 데코레이터 추가   │
│  2. JSON 스키마 수동 작성      │     → 스키마 자동 생성!            │
│  3. handle_tool_calls() 구현   │     → 호출도 자동 처리!            │
│  4. while 루프에서 직접 관리   │     → Runner가 자동 관리!          │
│                                │                                    │
│  약 30줄                       │  약 5줄                            │
└────────────────────────────────┴────────────────────────────────────┘
```

In [ ]:
# @function_tool로 도구 정의 — 이메일 전송 시뮬레이션
# 실제 이메일 전송 대신 로깅으로 대체합니다

sent_emails = []  # 전송된 이메일을 저장할 리스트

@function_tool
def send_email(to: str, subject: str, body: str) -> Dict[str, str]:
    """영업 대상에게 이메일을 전송합니다."""
    email_record = {"to": to, "subject": subject, "body": body}
    sent_emails.append(email_record)
    print(f"\n{'='*50}")
    print(f"[이메일 전송됨]")
    print(f"수신: {to}")
    print(f"제목: {subject}")
    print(f"{'='*50}")
    print(body)
    print(f"{'='*50}\n")
    return {"status": "success", "message": f"{to}에게 이메일이 전송되었습니다."}

In [ ]:
# @function_tool이 자동으로 생성한 도구 정보를 확인해봅시다
# 이전에 JSON 스키마를 직접 작성했던 것과 비교해보세요!

print(f"도구 이름: {send_email.name}")
print(f"도구 설명: {send_email.description}")
print(f"도구 파라미터 스키마: {send_email.params_json_schema}")

> **참고**: 함수의 **docstring**이 자동으로 `description`이 되고, **타입 힌트**가 자동으로 JSON 스키마의 `type`이 됩니다. 이전에 직접 작성하던 모든 것이 자동화되었습니다!

---

## 5. 에이전트를 도구로 변환: as_tool()

Agents SDK의 강력한 기능 중 하나는 **에이전트 자체를 도구로 변환**할 수 있다는 것입니다.

### 왜 에이전트를 도구로 변환하는가?

Section 3에서는 `asyncio.gather()`로 에이전트들을 **우리 코드에서 직접** 병렬 실행하고, 결과를 모아서 다시 선택 에이전트에게 전달했습니다. 이 방식은 **개발자가 흐름을 하드코딩**하는 워크플로우입니다.

에이전트를 도구로 변환하면, **상위 에이전트(오케스트레이터)가 스스로 판단하여** 하위 에이전트를 호출할 수 있습니다. 이것이 진정한 에이전트 패턴입니다.

```
┌──────────────────────────────────────────────────────────────────────┐
│  [워크플로우] Section 3 방식 — 개발자가 흐름을 코드로 제어          │
│                                                                      │
│  Python 코드:                                                        │
│    results = await asyncio.gather(                                   │
│        Runner.run(agent1, msg),    ← 개발자가 직접 호출              │
│        Runner.run(agent2, msg),                                      │
│        Runner.run(agent3, msg),                                      │
│    )                                                                 │
│    best = await Runner.run(picker, results)  ← 개발자가 직접 전달    │
├──────────────────────────────────────────────────────────────────────┤
│  [에이전트] as_tool() 방식 — LLM이 흐름을 자율적으로 결정           │
│                                                                      │
│  매니저 에이전트가 스스로:                                            │
│    1. professional_agent 도구 호출  ← LLM이 판단                     │
│    2. humorous_agent 도구 호출      ← LLM이 판단                     │
│    3. concise_agent 도구 호출       ← LLM이 판단                     │
│    4. 결과 비교 후 최적 선택        ← LLM이 판단                     │
│    5. 불만족시 다시 호출 가능       ← LLM이 판단                     │
└──────────────────────────────────────────────────────────────────────┘
```

### as_tool()의 핵심 특징

| 특징 | 설명 |
|------|------|
| **제어가 돌아옴** | 핸드오프와 달리, 호출 후 결과를 받아서 추가 판단 가능 |
| **오케스트레이터 패턴** | 상위 에이전트가 여러 하위 에이전트를 호출하고 결과를 종합 |
| **함수 도구와 통합** | `@function_tool` 함수와 같은 `tools` 리스트에 함께 배치 가능 |
| **전문성 캡슐화** | 각 에이전트의 고유한 instructions가 유지되므로 역할 분리가 명확 |
| **동적 의사결정** | LLM이 어떤 에이전트를 호출할지, 몇 번 호출할지 스스로 결정 |

```
Agent.as_tool() → Tool 객체

┌──────────────┐     as_tool()      ┌──────────────┐
│  영업 에이전트 │ ──────────────▶  │  도구 객체    │
│  (Agent)      │                    │  (Tool)       │
└──────────────┘                    └──────────────┘

다른 에이전트가 이 도구를 호출하면:
  1. 원본 에이전트가 작업을 수행
  2. 결과를 호출한 에이전트에게 반환
  3. 제어가 호출한 에이전트로 복귀 ← 핵심!
```

In [ ]:
# 영업 에이전트를 도구로 변환

description = "콜드 영업 이메일을 작성합니다."

tool1 = sales_agent1.as_tool(tool_name="professional_agent", tool_description="전문적이고 격식 있는 " + description)
tool2 = sales_agent2.as_tool(tool_name="humorous_agent", tool_description="유머러스하고 친근한 " + description)
tool3 = sales_agent3.as_tool(tool_name="concise_agent", tool_description="간결하고 핵심만 담은 " + description)

# 도구 목록: 3개의 에이전트 도구 + 1개의 함수 도구
tools = [tool1, tool2, tool3, send_email]

for t in tools:
    print(f"  - {t.name}: {t.description[:50]}...")

---

## 6. 세일즈 매니저: 오케스트레이터 에이전트

이제 모든 도구를 조합하여 **세일즈 매니저** 에이전트를 만듭니다.

이것은 이전 노트북에서 배운 **Orchestrator-Workers** 패턴의 실제 구현입니다!

```
┌──────────────────────────────────────────────────────────────────┐
│                      세일즈 매니저                               │
│                                                                  │
│                  ┌─────────────────┐                            │
│                  │  세일즈 매니저   │                            │
│                  │  (Orchestrator) │                            │
│                  └────────┬────────┘                            │
│                           │                                      │
│           ┌───────────────┼───────────────┐                      │
│           ▼               ▼               ▼                      │
│     ┌──────────┐   ┌──────────┐   ┌──────────┐                 │
│     │ 전문적   │   │ 유머     │   │ 간결     │  ← 에이전트     │
│     │ 에이전트 │   │ 에이전트 │   │ 에이전트 │    도구 호출     │
│     └──────────┘   └──────────┘   └──────────┘                 │
│           │               │               │                      │
│           └───────────────┼───────────────┘                      │
│                           ▼                                      │
│                    최적 이메일 선택                               │
│                           │                                      │
│                           ▼                                      │
│                    ┌──────────┐                                  │
│                    │send_email│  ← 함수 도구 호출                │
│                    └──────────┘                                  │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

In [ ]:
manager_instructions = """
You are a Sales Manager at AiDesk. Your goal is to find the single best cold sales email and send it.

Follow these steps carefully:

1. Call each of the three tools ONCE: professional_agent, humorous_agent, concise_agent.
   Each generates a Korean cold sales email draft. Call all three tools in a single turn if possible.

2. After receiving all three drafts, pick the single best one. Do NOT call any sales_agent tools again.

3. Use the send_email tool to send the best email. Send exactly one email.

Rules:
- Call each sales_agent tool exactly once.
- Never regenerate or rewrite drafts.
- Always finish by sending exactly one email via send_email.
"""

sales_manager = Agent(
    name="sales_manager",
    instructions=manager_instructions,
    tools=tools,
    model="gpt-4o-mini"
)

print(f"에이전트: {sales_manager.name}")
print(f"도구 수: {len(sales_manager.tools)}")
print(f"사용 가능한 도구: {[t.name for t in sales_manager.tools]}")

In [ ]:
# 세일즈 매니저 실행!

sent_emails.clear()  # 이전 기록 초기화

message = "'대표님께' 로 시작하는 콜드 영업 이메일을 보내주세요. 발신자는 '김영업'입니다."

with trace("영업 매니저 파이프라인"):
    result = await Runner.run(sales_manager, message, max_turns=10)

print("\n" + "=" * 60)
print("[최종 결과]")
print("=" * 60)
print(result.final_output)

> **트레이스 확인**: https://platform.openai.com/traces 에서 "영업 매니저 파이프라인" 트레이스를 찾아보세요.
> 세일즈 매니저가 3개의 에이전트 도구를 호출하고, 평가 후 send_email을 호출하는 전체 흐름을 시각적으로 확인할 수 있습니다.

---

## 7. Handoff: 에이전트 간 제어 전달

**도구(Tool)**는 호출 후 제어가 원래 에이전트로 돌아오지만, **핸드오프(Handoff)**는 제어를 완전히 다른 에이전트로 넘깁니다.

이번에는 핸드오프를 활용하여 더 정교한 파이프라인을 만들어봅시다.

```
┌──────────────────────────────────────────────────────────────────┐
│                  도구 vs 핸드오프 비교                           │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  [도구] 세일즈 매니저 ──호출──▶ 영업 에이전트 ──결과──▶ 매니저  │
│         (제어 유지)      ↑                               │      │
│                          └───────────────────────────────┘      │
│                                                                  │
│  [핸드오프] 세일즈 매니저 ──위임──▶ 이메일 매니저               │
│             (제어 종료)              (제어 인수)                  │
│                                      ├──▶ 제목 작성             │
│                                      ├──▶ HTML 변환             │
│                                      └──▶ 이메일 전송           │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

In [ ]:
# 이메일 전송 파이프라인의 보조 에이전트들을 정의합니다

# 제목 작성 에이전트 (도구로 사용)
subject_writer = Agent(
    name="이메일 제목 작성기",
    instructions="""콜드 영업 이메일의 제목을 작성합니다.
이메일 본문을 받으면 열어보고 싶은 매력적인 제목을 작성하세요.
제목만 출력하세요.""",
    model="gpt-4o-mini"
)
subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="콜드 영업 이메일의 제목을 작성합니다."
)

# HTML 변환 에이전트 (도구로 사용)
html_converter = Agent(
    name="HTML 이메일 변환기",
    instructions="""텍스트 이메일 본문을 HTML 이메일로 변환합니다.
마크다운이 포함된 텍스트 이메일을 받으면 깔끔하고 전문적인 HTML 이메일로 변환하세요.
인라인 CSS를 사용하여 보기 좋게 디자인하세요.""",
    model="gpt-4o-mini"
)
html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="텍스트 이메일 본문을 HTML 이메일로 변환합니다."
)

In [ ]:
# HTML 이메일 전송 도구

@function_tool
def send_html_email(to: str, subject: str, html_body: str) -> Dict[str, str]:
    """제목과 HTML 본문으로 이메일을 전송합니다."""
    email_record = {"to": to, "subject": subject, "html_body": html_body}
    sent_emails.append(email_record)
    print(f"\n{'='*50}")
    print(f"[HTML 이메일 전송됨]")
    print(f"수신: {to}")
    print(f"제목: {subject}")
    print(f"{'='*50}")
    print(html_body[:500] + "..." if len(html_body) > 500 else html_body)
    print(f"{'='*50}\n")
    return {"status": "success", "message": f"{to}에게 HTML 이메일이 전송되었습니다."}

### 왜 이메일 매니저를 핸드오프로 만드는가?

Section 6에서는 `send_email`을 세일즈 매니저의 도구로 넣어서 **매니저가 직접 전송**했습니다. 이번에는 전송 프로세스를 별도 에이전트에게 **핸드오프**합니다. 왜 이렇게 바꿀까요?

#### 1. 세일즈 매니저가 결과를 돌려받을 필요가 없다

세일즈 매니저의 역할은 **최적 이메일을 선택**하는 것까지입니다. 전송 결과를 받아서 추가 판단할 일이 없습니다. 이런 경우 제어를 완전히 넘기는 **핸드오프**가 적합합니다.

```
도구를 쓸 때:  매니저 → 전송 → 결과 돌아옴 → 매니저가 또 뭔가 해야 함
핸드오프:      매니저 → 전송 에이전트에게 위임 → 매니저는 끝!  ← 더 자연스러움
```

#### 2. 관심사 분리 (Separation of Concerns)

전송 프로세스에는 **제목 작성 → HTML 변환 → 전송**이라는 자체 파이프라인이 있습니다. 이것을 매니저의 도구로 넣으면 매니저가 6개 도구(영업 에이전트 3개 + 제목 + HTML + 전송)를 관리해야 합니다.

```
┌──────────────────────────────────────────────────────────────────┐
│  [도구 방식] 매니저가 모든 것을 직접 관리                         │
│                                                                  │
│  매니저의 tools: [agent1, agent2, agent3,                        │
│                  subject_writer, html_converter, send_html_email]│
│                                                                  │
│  → 도구가 6개로 복잡해짐                                         │
│  → 매니저 instructions에 전송 절차까지 설명해야 함               │
│  → 역할이 "이메일 선택 + 포맷팅 + 전송"으로 비대해짐             │
├──────────────────────────────────────────────────────────────────┤
│  [핸드오프 방식] 각자 역할에 집중                                │
│                                                                  │
│  매니저의 tools: [agent1, agent2, agent3]                        │
│  매니저의 handoffs: [email_manager]                              │
│                                                                  │
│  → 매니저는 "선택"에만 집중                                      │
│  → 이메일 매니저는 "포맷팅 + 전송"에만 집중                      │
│  → 각 에이전트의 instructions가 단순하고 명확                    │
└──────────────────────────────────────────────────────────────────┘
```

#### 3. 실무에서의 핸드오프 활용 예시

| 시나리오 | 핸드오프 대상 | 이유 |
|----------|-------------|------|
| 고객 문의 → 전문 상담 | 기술지원 에이전트 | 분류 후 제어를 넘기면 됨 |
| 주문 접수 → 결제 처리 | 결제 에이전트 | 주문 에이전트가 결제 결과를 처리할 필요 없음 |
| 콘텐츠 작성 → 퍼블리싱 | 배포 에이전트 | 작성자가 배포 세부사항을 알 필요 없음 |
| **이메일 선택 → 전송** | **이메일 매니저** | **매니저가 전송 세부사항을 알 필요 없음** |

> **핵심 판단 기준**: 결과를 돌려받아서 **추가 판단이 필요한가?** Yes → `as_tool()`, No → `handoff`

#### 주의: 핸드오프 에이전트의 name은 영문으로!

SDK는 핸드오프 에이전트의 `name`으로 `transfer_to_{name}` 형태의 tool name을 자동 생성합니다. **한글 name을 사용하면** `transfer_to________`로 변환되어 LLM이 핸드오프를 인식하지 못합니다.

```python
# ❌ 한글 name → transfer_to________  (LLM이 인식 불가)
Agent(name="이메일 매니저", ...)

# ✅ 영문 name → transfer_to_email_manager  (정상 동작)
Agent(name="email_manager", ...)
```

> **팁**: `name`은 영문으로, `instructions`는 한국어로 작성하세요. `name`은 내부 식별자이고 `instructions`가 에이전트의 실제 행동을 결정합니다.

In [33]:
# 이메일 매니저 에이전트 — 핸드오프 대상
# 이메일 본문을 받아서 제목 작성 → HTML 변환 → 전송까지 처리합니다
#
# 주의: handoff 대상 에이전트의 name은 반드시 영문으로!
# SDK가 name으로 "transfer_to_{name}" 형태의 tool name을 자동 생성하는데,
# 한글이 포함되면 "transfer_to________"로 변환되어 LLM이 인식하지 못합니다.

emailer_agent = Agent(
    name="email_manager",
    instructions="""당신은 이메일 포맷팅 및 전송을 담당합니다.
이메일 본문을 받으면 다음 순서로 처리하세요:
1. subject_writer 도구로 이메일 제목을 작성
2. html_converter 도구로 본문을 HTML로 변환
3. send_html_email 도구로 이메일을 전송

수신자 주소가 명시되지 않은 경우 'prospect@example.com'을 사용하세요.""",
    tools=[subject_tool, html_tool, send_html_email],
    model="gpt-4o-mini",
    handoff_description="Format the email as HTML and send it."
)

In [ ]:
# 확인: 도구와 핸드오프 목록

agent_tools = [tool1, tool2, tool3]  # 에이전트 도구 (이메일 생성)
handoffs = [emailer_agent]            # 핸드오프 대상 (이메일 포맷팅+전송)

print("[에이전트 도구 — 호출 후 제어 복귀]")
for t in agent_tools:
    print(f"  - {t.name}")

print("\n[핸드오프 — 제어 전달]")
for h in handoffs:
    print(f"  - {h.name}: {h.handoff_description}")

---

## 8. 완전한 파이프라인: 세일즈 매니저 + 핸드오프

이제 **도구**와 **핸드오프**를 결합한 완전한 영업 자동화 파이프라인을 구축합니다.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    완전한 영업 자동화 파이프라인                      │
│                                                                      │
│  ┌────────────────────────────────────────────────┐                  │
│  │              세일즈 매니저                       │                  │
│  │                                                 │                  │
│  │  1. 에이전트 도구 호출 (제어 복귀)               │                  │
│  │     ├─ professional_agent → 이메일 A             │                  │
│  │     ├─ humorous_agent → 이메일 B                 │                  │
│  │     └─ concise_agent → 이메일 C                  │                  │
│  │                                                 │                  │
│  │  2. 최적 이메일 선택                             │                  │
│  │                                                 │                  │
│  │  3. 핸드오프 (제어 전달) ─────────────────────┐  │                  │
│  └────────────────────────────────────────────── │──┘                  │
│                                                  │                    │
│                                                  ▼                    │
│  ┌────────────────────────────────────────────────┐                  │
│  │              이메일 매니저                       │                  │
│  │                                                 │                  │
│  │  4. subject_writer → 제목 생성                  │                  │
│  │  5. html_converter → HTML 변환                  │                  │
│  │  6. send_html_email → 이메일 전송               │                  │
│  └────────────────────────────────────────────────┘                  │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
```

### 워크플로우 → 에이전트 전환의 핵심

이전 노트북에서 배운 Anthropic의 정의를 기억하시나요?

- **워크플로우**: 미리 정의된 코드 경로로 실행
- **에이전트**: LLM이 자체적으로 프로세스와 도구 사용을 결정

이 파이프라인에서 **세일즈 매니저가 어떤 이메일을 선택할지는 LLM이 결정**합니다.
결과에 만족하지 않으면 도구를 다시 호출할 수도 있습니다.
이것이 단순한 워크플로우가 아닌 **에이전트**인 이유입니다.

In [34]:
# 완전한 세일즈 매니저 에이전트 (도구 + 핸드오프)

full_manager_instructions = """
You are a Sales Manager at AiDesk. Your goal is to find the single best cold sales email and send it.

Follow these steps carefully:

1. Call each of the three tools ONCE: professional_agent, humorous_agent, concise_agent.
   Each generates a Korean cold sales email draft. Call all three tools in a single turn if possible.

2. After receiving all three drafts, pick the single best one. Do NOT call any sales_agent tools again.

3. Immediately hand off the winning email text to email_manager using transfer_to_email_manager.

Rules:
- Call each sales_agent tool exactly once.
- Never regenerate or rewrite drafts.
- Always finish by handing off to email_manager.
"""

sales_manager_v2 = Agent(
    name="sales_manager_v2",
    instructions=full_manager_instructions,
    tools=agent_tools,       # 에이전트 도구 (이메일 생성)
    handoffs=handoffs,       # 핸드오프 (이메일 포맷팅+전송)
    model="gpt-4o-mini"
)

print(f"에이전트: {sales_manager_v2.name}")
print(f"도구: {[t.name for t in sales_manager_v2.tools]}")
print(f"핸드오프: {[h.name for h in sales_manager_v2.handoffs]}")

에이전트: sales_manager_v2
도구: ['professional_agent', 'humorous_agent', 'concise_agent']
핸드오프: ['email_manager']


In [35]:
# 완전한 파이프라인 실행!
# max_turns: 에이전트의 최대 실행 턴 수를 제한하여 무한 루프 방지

sent_emails.clear()  # 이전 기록 초기화

message = "'CTO님께' 로 시작하는 콜드 영업 이메일을 보내주세요. 발신자는 '박서연'입니다."

with trace("자동화 영업 파이프라인"):
    result = await Runner.run(sales_manager_v2, message, max_turns=10)

print("\n" + "=" * 60)
print("[파이프라인 실행 완료]")
print("=" * 60)
print(f"최종 에이전트: {result.last_agent.name}")
print(f"최종 출력:\n{result.final_output}")


[HTML 이메일 전송됨]
수신: prospect@example.com
제목: 고객 서비스 혁신, 귀사에 맞는 맞춤형 솔루션 제안!
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AiDesk 소개 이메일</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: #f4f4f4;
        }
        .container {
            background-color: #ffffff;
            border-radius: 8px;
            padding: 20px;
            max-width: 600px;
      ...


[파이프라인 실행 완료]
최종 에이전트: email_manager
최종 출력:
이메일이 성공적으로 **prospect@example.com**에게 전송되었습니다. 추가적으로 도움이 필요하시면 언제든지 말씀해 주세요!


### Trace 확인

https://platform.openai.com/traces 에서 **"자동화 영업 파이프라인"** 트레이스를 확인해보세요.

다음 흐름을 시각적으로 볼 수 있습니다:
1. 영업 매니저가 3개의 에이전트 도구 호출
2. 최적 이메일 선택
3. 이메일 매니저에게 핸드오프
4. 이메일 매니저가 제목 작성, HTML 변환, 전송 순서로 처리

---

## 9. 전송된 이메일 확인

In [36]:
# 전송된 이메일 기록 확인

print(f"총 전송된 이메일 수: {len(sent_emails)}\n")

for i, email in enumerate(sent_emails, 1):
    print(f"--- 이메일 #{i} ---")
    print(f"수신: {email.get('to', 'N/A')}")
    print(f"제목: {email.get('subject', 'N/A')}")
    if 'html_body' in email:
        print(f"형식: HTML")
        print(f"본문 미리보기: {email['html_body'][:200]}...")
    else:
        print(f"형식: 텍스트")
        print(f"본문: {email.get('body', 'N/A')[:200]}...")
    print()

총 전송된 이메일 수: 1

--- 이메일 #1 ---
수신: prospect@example.com
제목: 고객 서비스 혁신, 귀사에 맞는 맞춤형 솔루션 제안!
형식: HTML
본문 미리보기: <!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AiDesk 소개 이메일</title>
    <style>
        body ...



In [37]:
# HTML 이메일이 있다면 Jupyter에서 렌더링해봅시다

from IPython.display import HTML, display

html_emails = [e for e in sent_emails if 'html_body' in e]
if html_emails:
    print(f"HTML 이메일 {len(html_emails)}개를 렌더링합니다:\n")
    for email in html_emails:
        print(f"제목: {email['subject']}")
        display(HTML(email['html_body']))
else:
    print("HTML 이메일이 없습니다.")

HTML 이메일 1개를 렌더링합니다:

제목: 고객 서비스 혁신, 귀사에 맞는 맞춤형 솔루션 제안!


---

## 10. 에이전트 디자인 패턴 정리

이번 노트북에서 사용한 패턴들을 정리해봅시다:

### 사용된 디자인 패턴

| 패턴 | 위치 | 설명 |
|------|------|------|
| **Parallelization (병렬화)** | Section 3.2 | 3개의 에이전트를 `asyncio.gather()`로 병렬 실행 |
| **Voting (투표)** | Section 3.3 | 여러 결과 중 최적 선택 |
| **Orchestrator-Workers** | Section 6 | 매니저가 하위 에이전트에게 작업 분배 |
| **Prompt Chaining** | Section 7-8 | 이메일 매니저의 순차 처리 (제목→HTML→전송) |
| **Tool Use** | Section 4-6 | `@function_tool`과 `as_tool()` |

### 워크플로우 vs 에이전트

```
┌─────────────────────────────────────────────────────────────────────┐
│  이 시스템이 "에이전트"인 이유:                                     │
│                                                                     │
│  세일즈 매니저가 결과에 만족하지 않으면 도구를 다시 호출할 수 있음   │
│  → LLM이 프로세스의 진행을 동적으로 결정                            │
│  → 단순한 고정 경로 워크플로우가 아님                               │
│                                                                     │
│  이메일 매니저는 "워크플로우"에 가까움:                              │
│  → 제목 작성 → HTML 변환 → 전송이라는 고정된 순서를 따름            │
│                                                                     │
│  실제 시스템은 워크플로우와 에이전트를 적절히 조합합니다              │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 11. 연습 과제 및 확장 아이디어

### 연습 과제

1. **에이전트 추가**: 새로운 스타일의 영업 에이전트를 추가해보세요 (예: 데이터 중심, 스토리텔링 등)
2. **평가 기준 강화**: `sales_picker`에 구체적인 평가 기준(수신자 관점, CTA 명확성 등)을 추가해보세요
3. **실제 이메일 연동**: SendGrid나 Resend를 연동하여 실제 이메일을 발송해보세요

### 확장 아이디어

| 아이디어 | 설명 |
|----------|------|
| 수신자 조사 에이전트 | 웹 검색 도구로 수신자 정보를 조사하고 맞춤 이메일 작성 |
| A/B 테스트 시스템 | 여러 버전의 이메일 성과를 추적하고 최적화 |
| 팔로업 에이전트 | 응답이 없을 때 자동으로 후속 이메일 작성 |
| CRM 연동 | 고객 정보 DB와 연동하여 개인화된 이메일 작성 |
| 가드레일 에이전트 | 이메일 내용의 적절성/컴플라이언스를 병렬로 검증 |

---

## 참고 자료

- [OpenAI Agents SDK 문서](https://openai.github.io/openai-agents-python/)
- [OpenAI Agents SDK GitHub](https://github.com/openai/openai-agents-python)
- [OpenAI Traces 대시보드](https://platform.openai.com/traces)
- [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/) — 유사한 패턴, 다른 프레임워크

### 참고: Google ADK 코드 예시

Google의 ADK도 OpenAI Agents SDK와 매우 유사한 패턴을 따릅니다:

```python
# Google ADK
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="시간과 날씨에 대한 질문에 답하는 에이전트",
    instruction="도시의 시간과 날씨에 대한 사용자 질문에 답하는 도움이 되는 에이전트입니다.",
    tools=[get_weather, get_current_time]
)
```

에이전트 프레임워크들은 공통적으로 **Agent + Tools + Instructions** 패턴을 채택하고 있습니다.